In [2]:
%%bigquery
CREATE SCHEMA IF NOT EXISTS weather_lab;

Query is running:   0%|          |

""


In [3]:
%%bigquery
CREATE OR REPLACE TABLE weather_lab.weather_data (
  date DATE,
  location STRING,
  temperature_c FLOAT64,
  humidity FLOAT64,
  wind_speed_kph FLOAT64,
  precipitation_mm FLOAT64,
  condition STRING
);

Query is running:   0%|          |

""


In [4]:
%%bigquery
LOAD DATA OVERWRITE weather_lab.weather_data
FROM FILES (
  format = 'CSV',
  uris = ['gs://labs.roitraining.com/data-to-ai-workshop/weather_data.csv'],
  skip_leading_rows = 1
);

Query is running:   0%|          |

""


In [5]:
%%bigquery
SELECT *
FROM weather_lab.weather_data
LIMIT 10;

Query is running:   0%|          |

Downloading:   0%|          |

,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition
0,2025-02-21,Atlanta,GA,55.7,5.0,0.12,29.80,50.4,Cloudy
1,2025-02-26,Atlanta,GA,75.2,10.4,0.03,29.58,49.9,Cloudy
2,2025-03-01,Atlanta,GA,51.7,4.7,0.08,29.74,49.9,Cloudy
3,2025-03-05,Atlanta,GA,74.4,5.1,0.02,29.92,50.4,Cloudy
4,2025-03-10,Atlanta,GA,59.5,9.6,0.09,29.67,57.2,Cloudy
5,2025-03-14,Atlanta,GA,71.7,7.2,0.18,29.92,55.3,Cloudy
6,2025-02-19,Boston,MA,61.7,3.9,0.11,29.62,54.1,Cloudy
7,2025-03-09,Boston,MA,76.7,4.3,0.09,29.52,40.9,Cloudy
8,2025-03-13,Boston,MA,71.9,9.8,0.16,29.99,42.3,Cloudy
9,2025-03-19,Boston,MA,60.7,6.4,0.04,29.83,49.4,Cloudy


In [7]:
%%bigquery
SELECT
  city,
  state,
  AVG(temperature_f) AS avg_temp_f,
  AVG(precipitation_in) AS avg_precip_in
FROM weather_lab.weather_data
GROUP BY city, state;

Query is running:   0%|          |

Downloading:   0%|          |

,city,state,avg_temp_f,avg_precip_in
0,Atlanta,GA,58.283333,0.941333
1,Boston,MA,56.906667,0.922667
2,Chicago,IL,48.850000,0.949667
3,Denver,CO,55.616667,1.041667
4,Houston,TX,52.196667,0.615667
5,Los Angeles,CA,52.496667,1.094667
6,Miami,FL,49.193333,0.927000
7,New York,NY,58.913333,0.775000
8,San Francisco,CA,53.320000,0.887667
9,Seattle,WA,48.406667,1.145000


In [25]:
%%bigquery
CREATE OR REPLACE MODEL `weather_lab.gemini_model` REMOTE WITH CONNECTION DEFAULT OPTIONS (ENDPOINT = 'gemini-2.5-flash');


Query is running:   0%|          |

""


In [ ]:
# dataframe:
# uuid: 0648FAE7-1574-4A8B-9497-4F2B80DCA30F
# output_variable:
# config_str:

import google.colabsqlviz.explore_dataframe as _vizcell
_vizcell.explore_dataframe(df_or_df_name='', uuid='0648FAE7-1574-4A8B-9497-4F2B80DCA30F')

In [31]:
%%bigquery
CREATE OR REPLACE TABLE `weather_lab.weather_reports` AS
SELECT
  ml_generate_text_result['candidates'][0]['content']['parts'][0]['text'] AS weather_report,
  date,
  city,
  state,
  temperature_f,
  wind_speed_mph,
  precipitation_in,
  barometric_pressure_inHg,
  humidity_percent,
  weather_condition,
  ml_generate_text_status AS api_status
FROM ML.GENERATE_TEXT(
  MODEL `weather_lab.gemini_model`,
  (
    SELECT
      CONCAT(
        'Create a short public weather report or warning for the next 6-12 hours. ',
        'City: ', city, ', State: ', state, '. ',
        'Temperature: ', CAST(temperature_f AS STRING), '°F. ',
        'Wind: ', CAST(wind_speed_mph AS STRING), ' mph. ',
        'Precipitation: ', CAST(precipitation_in AS STRING), ' inches. ',
        'Pressure: ', CAST(barometric_pressure_inHg AS STRING), ' inHg. ',
        'Humidity: ', CAST(humidity_percent AS STRING), '%. ',
        'Condition: ', weather_condition, '. ',
        'If a warning is needed, state it clearly; otherwise say “No warning.”'
      ) AS prompt,
      date, city, state, temperature_f, wind_speed_mph, precipitation_in,
      barometric_pressure_inHg, humidity_percent, weather_condition
    FROM `weather_lab.weather_data`
    LIMIT 50
  ),
  STRUCT(0.2 AS temperature, 256 AS max_output_tokens)
);

Query is running:   0%|          |

""


In [32]:
%%bigquery
SELECT city, state, date, weather_report
FROM `weather_lab.weather_reports`
LIMIT 5;

Query is running:   0%|          |

Downloading:   0%|          |

,city,state,date,weather_report
0,Atlanta,GA,2025-03-10,"""Here's your Atlanta weather update for the ne..."
1,Atlanta,GA,2025-03-14,"""Here's your Atlanta weather update for the ne..."
2,Atlanta,GA,2025-02-21,"""**Atlanta Weather Update: Cloudy and Mild Ove..."
3,Atlanta,GA,2025-03-05,"""**Atlanta Weather Update: Cloudy with a Chanc..."
4,Atlanta,GA,2025-02-26,"""Here's your Atlanta weather update for the ne..."


In [39]:
%%bigquery
CREATE OR REPLACE TABLE `weather_lab.weather_reports` AS
SELECT
  date,
  city,
  state,
  temperature_f,
  wind_speed_mph,
  precipitation_in,
  barometric_pressure_inHg,
  humidity_percent,
  weather_condition,
  AI.GENERATE(
    CONCAT(
      'Create a short public weather report or warning for the next 6 to 12 hours. ',
      'City: ', city, ', State: ', state, '. ',
      'Temperature: ', CAST(temperature_f AS STRING), ' degrees Fahrenheit. ',
      'Wind: ', CAST(wind_speed_mph AS STRING), ' miles per hour. ',
      'Precipitation: ', CAST(precipitation_in AS STRING), ' inches. ',
      'Pressure: ', CAST(barometric_pressure_inHg AS STRING), ' inHg. ',
      'Humidity: ', CAST(humidity_percent AS STRING), ' percent. ',      'Condition: ', weather_condition, '. ',
      'If a warning is needed, state it clearly. Otherwise say no warning.'
    ),
    endpoint => 'gemini-2.5-flash',
    model_params => JSON "{ \"generationConfig\": { \"temperature\": 0.2, \"maxOutputTokens\": 256 } }"
  ).result AS weather_report
FROM `weather_lab.weather_data`;

Query is running:   0%|          |

""


In [40]:
%%bigquery
SELECT
  city,
  state,
  date,
  weather_report
FROM `weather_lab.weather_reports`
LIMIT 10;

Query is running:   0%|          |

Downloading:   0%|          |

,city,state,date,weather_report
0,Atlanta,GA,2025-03-01,None
1,Atlanta,GA,2025-03-10,None
2,Atlanta,GA,2025-02-26,None
3,Atlanta,GA,2025-03-05,"Here's your weather report for Atlanta, GA:"
4,Atlanta,GA,2025-03-14,None
5,Atlanta,GA,2025-02-21,**Atlanta Weather Report - Next
6,Boston,MA,2025-03-09,None
7,Boston,MA,2025-03-13,None
8,Boston,MA,2025-03-19,None
9,Boston,MA,2025-02-19,**Boston Weather Report - Next 6 to


In [42]:
%%bigquery
SELECT
  COUNT(*) AS total_weather_rows
FROM `weather_lab.weather_data`;




Query is running:   0%|          |

Downloading:   0%|          |

,total_weather_rows
0,300


In [44]:
%%bigquery
SELECT
  COUNT(*) AS total_generated_reports
FROM `weather_lab.weather_reports`;

Query is running:   0%|          |

Downloading:   0%|          |

,total_generated_reports
0,300


In [45]:
%%bigquery
SELECT
  city,
  state,
  weather_condition,
  weather_report
FROM `weather_lab.weather_reports`
WHERE LOWER(weather_report) LIKE '%warning%'
LIMIT 10;

Query is running:   0%|          |

Downloading:   0%|          |

,city,state,weather_condition,weather_report
0,Los Angeles,CA,Rainy,**Los Angeles Weather Report & Warning**\n\n**
1,Los Angeles,CA,Rainy,**Public Weather Report & Warning
2,Seattle,WA,Rainy,**Seattle Weather Report & Warning**
3,Los Angeles,CA,Snowy,**Public Weather Report and Warning for Los
4,Miami,FL,Snowy,"**URGENT WEATHER WARNING FOR MIAMI, FLORIDA!**..."
5,Miami,FL,Snowy,**URGENT WEATHER WARNING
6,San Francisco,CA,Snowy,"**Public Weather Warning for San Francisco, CA..."
7,San Francisco,CA,Snowy,**San Francisco Weather Report & Warning**\n\n...
8,Seattle,WA,Snowy,**Seattle Weather Report & Warning**\n\n**
9,Seattle,WA,Snowy,**Seattle Weather Report & Warning**\n\n**


In [46]:
%%bigquery
SELECT *
FROM `weather_lab.weather_reports`
LIMIT 5;

Query is running:   0%|          |

Downloading:   0%|          |

,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition,weather_report
0,2025-03-01,Atlanta,GA,51.7,4.7,0.08,29.74,49.9,Cloudy,None
1,2025-03-10,Atlanta,GA,59.5,9.6,0.09,29.67,57.2,Cloudy,None
2,2025-02-26,Atlanta,GA,75.2,10.4,0.03,29.58,49.9,Cloudy,None
3,2025-03-05,Atlanta,GA,74.4,5.1,0.02,29.92,50.4,Cloudy,"Here's your weather report for Atlanta, GA:"
4,2025-03-14,Atlanta,GA,71.7,7.2,0.18,29.92,55.3,Cloudy,None
